# Forecast versus actual: US retail sales

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toclocoinc/lattice-grid-notebooks/blob/main/retail-forecast/retail_forecast.ipynb)

Fit an exponential-smoothing (ETS) forecast to a public FRED series, hold back the last two years, and compare **forecast, actual and residual** in a grid with a **residual colour scale**. A **fan chart** and a **seasonal decomposition chart** are bound to the grid: pick a period, filter the dates or just **select rows** in the grid and both charts redraw over them.

**Data:** [FRED series RSXFSN](https://fred.stlouisfed.org/series/RSXFSN), Advance Retail Sales: Retail Trade, monthly, millions of dollars, *not* seasonally adjusted. Source: U.S. Census Bureau, via the Federal Reserve Bank of St. Louis (FRED). Census Bureau data is in the public domain; FRED asks that you cite it. It is downloaded from FRED's keyless CSV endpoint when you run the notebook. The notebook code is MIT-licensed (see the repo `LICENSE`).

In [1]:
%pip install -q "lattice-grid-jupyter>=1.111.3.0" statsmodels

## 1. Fetch the series and fit a forecast

We keep 2012 onward, fit a damped-trend ETS model with multiplicative seasonality on everything **except the last 24 months**, then ask it for a fitted/forecast value and an 80% prediction interval for every month. The last 24 rows are a true out-of-sample forecast.

In [2]:
import warnings
import pandas as pd
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings("ignore")
FRED = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=RSXFSN"
sales = pd.read_csv(FRED, index_col=0, parse_dates=True).RSXFSN["2012":].astype(float)
sales.index.freq = "MS"

HOLDOUT = 24
fit = ETSModel(sales[:-HOLDOUT], error="mul", trend="add", damped_trend=True, seasonal="mul", seasonal_periods=12).fit(disp=False)
pred = fit.get_prediction(start=0, end=len(sales) - 1).summary_frame(alpha=0.2)
print(f"{len(sales)} months, {sales.index[0]:%b %Y} to {sales.index[-1]:%b %Y}; forecasting the last {HOLDOUT}")

177 months, Jan 2012 to Sep 2026; forecasting the last 24


## 2. The forecast frame

One row per month: `actual`, `forecast`, the 80% interval (`lower`, `upper`) and the `residual` (actual minus forecast). The `trend`, `seasonal` and `remainder` columns come from an STL decomposition of the actuals and feed the second chart. `period` says whether the row was used to fit the model or is a genuine forecast.

In [3]:
stl = STL(sales, period=12, robust=True).fit()
frame = pd.DataFrame({
    "date": sales.index.strftime("%Y-%m"), "period": ["fitted"] * (len(sales) - HOLDOUT) + ["forecast"] * HOLDOUT,
    "actual": sales.values, "forecast": pred["mean"].values, "lower": pred.pi_lower.values, "upper": pred.pi_upper.values,
    "trend": stl.trend.values, "seasonal": stl.seasonal.values, "remainder": stl.resid.values,
}).round(0)
frame["residual"] = frame.actual - frame.forecast
frame.insert(1, "year", frame.date.str[:4])
frame.tail(3)

,date,year,period,actual,forecast,lower,upper,trend,seasonal,remainder,residual
174,2026-07,2026,forecast,655940.0,610435.0,540984.0,684351.0,640454.0,16256.0,-770.0,45505.0
175,2026-08,2026,forecast,655885.0,620977.0,548795.0,701073.0,643049.0,13575.0,-739.0,34908.0
176,2026-09,2026,forecast,635056.0,579805.0,509218.0,649928.0,645654.0,-11941.0,1343.0,55251.0


## 3. Grid, fan chart and decomposition chart

The `residual` column has a **diverging colour scale**: blue where sales came in above the forecast, red where they fell short. Both charts are **bound to the grid**, so they follow the grid's filters, quick search and **selection** in the browser with no round trip to Python. Three ways to point them at a period:

- **Select rows.** Click a month, then Shift-click another to extend the selection: the fan and the decomposition redraw over just the selected months, and show everything again when you clear the selection. Try the first month of 2020 to the last of 2021.
- **Pick a period** in the dropdown (try `year = 2020`, or `period = forecast`). Select `All months` to see the full history.
- **Filter** with the grid's own column filters (hover a header).

In [4]:
from ipywidgets import Dropdown, HBox, Layout, VBox
from lattice_grid_jupyter import LatticeGridWidget, LatticeChart

scale = {"scale": {"min": -60000, "max": 60000, "colours": ["#d64545", "#ffffff", "#2f6fdb"]}}
grid = LatticeGridWidget(frame, height=420, options={"formatting": {"residual": [scale]}},
    columns=[{"field": "date", "layout": {"width": 70}}, {"field": "period", "layout": {"width": 70}}]
    + [{"field": f, "layout": {"width": 78}, "typeOptions": {"decimals": 0}} for f in ("actual", "forecast", "lower", "upper", "residual")]
    + [{"field": f, "layout": {"hidden": True}} for f in ("year", "trend", "seasonal", "remainder")])
fan = LatticeChart(grid, type="fan", x="date", y="actual", forecast="forecast", lower="lower", upper="upper", height=300)
decomp = LatticeChart(grid, type="decomposition", x="date", observed="actual", trend="trend", seasonal="seasonal", residual="remainder", height=300)
grid.layout, fan.layout, decomp.layout = Layout(width="58%", min_width="340px"), Layout(width="40%", min_width="340px"), Layout(width="100%")

pick = Dropdown(options=["All months"] + [f"year = {y}" for y in sorted(frame.year.unique())] + ["period = forecast"], description="Period")
def zoom(change):  # a period is just a grid filter: both bound charts redraw for it
    col, _, val = change.new.partition(" = ")
    grid.state = {**grid.state, "filters": {"op": "and", "conditions": [{"col": col, "op": "eq", "value": val}] if val else []}}
pick.observe(zoom, "value")
VBox([pick, HBox([grid, fan], layout=Layout(flex_flow="row wrap")), decomp])

![the grid with a range of months selected, the fan and decomposition charts redrawn over them](https://raw.githubusercontent.com/toclocoinc/lattice-grid-notebooks/main/retail-forecast/grid.png)

*The grid in JupyterLab with the months from January 2020 to December 2021 selected: the fan chart and the decomposition chart redraw over just those 24 months.*

## 4. Rows back as a DataFrame

`grid.selected` is whatever you picked as a pandas `DataFrame` (the grid's filtered view when nothing is selected). Here, how far off the forecast was, split into fitted and genuine forecast months:

In [5]:
rows = grid.selected if not grid.selected.empty else grid.view
rows.assign(pct_error=(rows.residual / rows.actual * 100).abs()).groupby("period").agg(
    months=("actual", "size"), mean_abs_pct_error=("pct_error", "mean"), outside_80pct=("actual", lambda a: ((a < rows.loc[a.index, "lower"]) | (a > rows.loc[a.index, "upper"])).sum())).round(2)

,months,mean_abs_pct_error,outside_80pct
period,,,
fitted,153,1.88,16
forecast,24,3.60,0
